In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

np.random.seed(42)

In [2]:
BASE_DIR = Path(
    "paper_implementation"
)

PROFILE_DIR = Path(
    "daily_profiles_24h"
)

MODE5_DIR = (
    BASE_DIR /
    "theft_simulation" /
    "mode_5"
)

MODE5_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
selected_df = pd.read_csv(
    BASE_DIR /
    "selected_150_meters" /
    "selected_150_meter_ids.csv"
)

fraud_df = pd.read_csv(
    BASE_DIR /
    "area_assignments" /
    "fraud_consumers.csv"
)

selected_meters = (
    selected_df["Meter"]
    .astype(str)
    .tolist()
)

fraud_meters = set(
    fraud_df["Meter"]
    .astype(str)
    .tolist()
)

print(
    "Selected Consumers:",
    len(selected_meters)
)

print(
    "Fraud Consumers:",
    len(fraud_meters)
)

Selected Consumers: 150
Fraud Consumers: 27


In [4]:
hour_cols = [
    f"HOUR_{i}"
    for i in range(24)
]

In [5]:
mode4_intervals = {}

for meter in fraud_meters:

    t1 = np.random.randint(
        0,
        22
    )

    t2 = np.random.randint(
        t1 + 2,
        24
    )

    mode4_intervals[meter] = (
        t1,
        t2
    )

print(
    list(
        mode4_intervals.items()
    )[:5]
)

[('42633', (6, 11)), ('16421', (14, 18)), ('45191', (7, 21)), ('14245', (20, 22)), ('10863', (18, 22))]


In [6]:
def apply_mode1(x_it):

    alpha_t = np.random.uniform(
        0.1,
        0.8
    )

    x_prime = (
        alpha_t
        * x_it
    )

    return {
        "x_prime": x_prime,
        "alpha_t": alpha_t,
        "Daily_Mean": "-",
        "Gamma": "-",
        "t1": "-",
        "t2": "-"
    }

In [7]:
def apply_mode2(
    daily_mean
):

    alpha_t = np.random.uniform(
        0.1,
        0.8
    )

    x_prime = (
        alpha_t
        * daily_mean
    )

    return {
        "x_prime": x_prime,
        "alpha_t": alpha_t,
        "Daily_Mean": daily_mean,
        "Gamma": "-",
        "t1": "-",
        "t2": "-"
    }

In [8]:
def apply_mode3(
    x_it,
    gamma
):

    x_prime = max(
        x_it - gamma,
        0
    )

    return {
        "x_prime": x_prime,
        "alpha_t": "-",
        "Daily_Mean": "-",
        "Gamma": gamma,
        "t1": "-",
        "t2": "-"
    }

In [9]:
def apply_mode4(
    x_it,
    hour_num,
    t1,
    t2
):

    if t1 < hour_num < t2:

        x_prime = 0

    else:

        x_prime = x_it

    return {
        "x_prime": x_prime,
        "alpha_t": "-",
        "Daily_Mean": "-",
        "Gamma": "-",
        "t1": t1,
        "t2": t2
    }

In [10]:
total_modified = 0

log_records = []

for meter in selected_meters:

    df = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    if meter in fraud_meters:

        t1, t2 = mode4_intervals[meter]

        for row_idx in df.index:

            date = df.loc[
                row_idx,
                "DATE"
            ]

            # Day number starts from 1

            day_number = row_idx + 1

            # Mode sequence:
            # 1 → 2 → 3 → 4 → repeat

            theft_mode = (
                ((day_number - 1) % 4)
                + 1
            )

            daily_mean = (
                df.loc[
                    row_idx,
                    hour_cols
                ]
                .astype(float)
                .mean()
            )

            daily_max = (
                df.loc[
                    row_idx,
                    hour_cols
                ]
                .astype(float)
                .max()
            )

            if daily_max == 0:

                gamma = 0

            else:

                gamma = np.random.uniform(
                    0,
                    daily_max
                )

            for hour in hour_cols:

                hour_num = int(
                    hour.split("_")[1]
                )

                x_it = float(
                    df.loc[
                        row_idx,
                        hour
                    ]
                )

                # -------------------------
                # MODE 1
                # -------------------------

                if theft_mode == 1:

                    result = apply_mode1(
                        x_it
                    )

                # -------------------------
                # MODE 2
                # -------------------------

                elif theft_mode == 2:

                    result = apply_mode2(
                        daily_mean
                    )

                # -------------------------
                # MODE 3
                # -------------------------

                elif theft_mode == 3:

                    result = apply_mode3(
                        x_it,
                        gamma
                    )

                # -------------------------
                # MODE 4
                # -------------------------

                else:

                    result = apply_mode4(
                        x_it,
                        hour_num,
                        t1,
                        t2
                    )

                x_prime = result[
                    "x_prime"
                ]

                # Save log

                log_records.append({

                    "Meter":
                    meter,

                    "Date":
                    date,

                    "Hour":
                    hour,

                    "Theft_Mode":
                    theft_mode,

                    "Actual_Reading":
                    x_it,

                    "alpha_t":
                    result["alpha_t"],

                    "Daily_Mean":
                    result["Daily_Mean"],

                    "Gamma":
                    result["Gamma"],

                    "t1":
                    result["t1"],

                    "t2":
                    result["t2"],

                    "Final_Output":
                    x_prime
                })

                # Replace reading

                df.loc[
                    row_idx,
                    hour
                ] = x_prime

                total_modified += 1

    df.to_csv(
        MODE5_DIR /
        f"{meter}.csv",
        index=False
    )

log_df = pd.DataFrame(
    log_records
)

log_df.to_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_5_generation_log.csv",
    index=False
)

print(
    "Modified readings:",
    total_modified
)

print(
    "Log rows:",
    len(log_df)
)

Modified readings: 20088
Log rows: 20088


In [11]:
print(
    "Files Generated:",
    len(
        list(
            MODE5_DIR.glob("*.csv")
        )
    )
)

log_df = pd.read_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_5_generation_log.csv"
)

print(
    "Log Rows:",
    len(log_df)
)

Files Generated: 150
Log Rows: 20088


In [12]:
normal_meters = [
    m for m in selected_meters
    if m not in fraud_meters
]

unchanged_count = 0

for meter in normal_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode5 = pd.read_csv(
        MODE5_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode5[hour_cols].values
    )

    if same:
        unchanged_count += 1

print(
    "Unchanged Normal Consumers:",
    unchanged_count,
    "/",
    len(normal_meters)
)

Unchanged Normal Consumers: 123 / 123


In [13]:
modified_count = 0

for meter in fraud_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode5 = pd.read_csv(
        MODE5_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode5[hour_cols].values
    )

    if not same:
        modified_count += 1

print(
    "Modified Fraud Consumers:",
    modified_count,
    "/",
    len(fraud_meters)
)

Modified Fraud Consumers: 27 / 27


In [14]:
sequence_check = []

for day in range(1, 32):

    expected_mode = (
        ((day - 1) % 4)
        + 1
    )

    actual_modes = log_df[
        log_df["Date"]
        ==
        f"2018-07-{day:02d}"
    ]["Theft_Mode"].unique()

    if len(actual_modes) > 0:

        sequence_check.append(
            expected_mode
            in actual_modes
        )

all(sequence_check)

True

In [17]:
log_df.head(100)

,Meter,Date,Hour,Theft_Mode,Actual_Reading,alpha_t,Daily_Mean,Gamma,t1,t2,Final_Output
0,6270,2018-07-01,HOUR_0,1,1.2354,0.5789631185585099,-,-,-,-,0.715251
1,6270,2018-07-01,HOUR_1,1,1.1880,0.408106745617721,-,-,-,-,0.484831
2,6270,2018-07-01,HOUR_2,1,0.8430,0.1854267643913452,-,-,-,-,0.156315
3,6270,2018-07-01,HOUR_3,1,0.5580,0.4466238370778892,-,-,-,-,0.249216
4,6270,2018-07-01,HOUR_4,1,0.5826,0.12407196478065288,-,-,-,-,0.072284
...,...,...,...,...,...,...,...,...,...,...,...
95,6270,2018-07-04,HOUR_23,4,2.2842,-,-,-,16,20,2.284200
96,6270,2018-07-05,HOUR_0,1,1.7778,0.1837159721568112,-,-,-,-,0.326610
97,6270,2018-07-05,HOUR_1,1,1.9992,0.5992713510560965,-,-,-,-,1.198063
98,6270,2018-07-05,HOUR_2,1,1.5744,0.6325495340318282,-,-,-,-,0.995886
